# CloneCharter — Inferencia completa desde test_dataset

Carga una canción de `test_dataset/`, corre el pipeline completo
(`ChartGenerationPipeline`) y guarda el `.chart` resultante.

Canciones disponibles:
- `Caos - La Planta`
- `El Precio de la Soledad`
- `Grupo Marca Registrada - El Rescate`

## 1 · Setup — paths e imports

In [1]:
#!pip install torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124

In [1]:
import sys, os, json, glob
from pathlib import Path

REPO_ROOT = Path("/s/mlopezs8/CloneCharter")
sys.path.insert(0, str(REPO_ROOT))

import torch

from inference.pipeline import ChartGenerationPipeline

print("CUDA available:", torch.cuda.is_available())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CUDA available: True


## 2 · Explorar canciones disponibles en test_dataset

In [2]:
TEST_DATASET_DIR = REPO_ROOT / "test_dataset"

songs = [d for d in TEST_DATASET_DIR.iterdir() if d.is_dir()]
print(f"Canciones en test_dataset: {len(songs)}\n")

def find_audio(song_dir: Path) -> Path | None:
    """Return the best available audio file (prefers separated stems)."""
    for name in ("song.ogg", "guitar.ogg", "song.mp3", "song.wav"):
        p = song_dir / name
        if p.exists():
            return p
    return None

def read_song_ini(song_dir: Path) -> dict:
    ini_path = song_dir / "song.ini"
    if not ini_path.exists():
        return {}
    info = {}
    for line in ini_path.read_text(errors="replace").splitlines():
        if "=" in line:
            k, _, v = line.partition("=")
            info[k.strip().lower()] = v.strip()
    return info

for i, song in enumerate(sorted(songs)):
    ini = read_song_ini(song)
    audio = find_audio(song)
    print(f"[{i}] {song.name}")
    print(f"     name   : {ini.get('name', 'N/A')}")
    print(f"     artist : {ini.get('artist', 'N/A')}")
    print(f"     audio  : {audio.name if audio else 'NOT FOUND'}")
    print()

Canciones en test_dataset: 3

[0] Caos - La Planta
     name   : La Planta
     artist : Caos
     audio  : song.ogg

[1] El Precio de la Soledad
     name   : El Precio de la Soledad
     artist : Alfredo Olivas
     audio  : song.ogg

[2] Grupo Marca Registrada - El Rescate
     name   : El Rescate (ft. Junior H)
     artist : Grupo Marca Registrada
     audio  : guitar.ogg



## 3 · Leer el `.chart` de referencia

Leemos el chart original para conocer el BPM, resolución y secciones disponibles, y para compararlo con el generado más adelante.

In [3]:
from data.chart_loader import CloneHeroChartParser

# ── Select a song (change SONG_IDX to try others) ────────────────────────────
SONG_IDX   = 0          # 0 = first alphabetically
INSTRUMENT = "Single"   # "Single" | "DoubleBass" | "Drums"
DIFFICULTY = "Expert"

song_dir   = sorted(songs)[SONG_IDX]
ini_info   = read_song_ini(song_dir)
audio_path = find_audio(song_dir)
chart_path = song_dir / "notes.chart"

print(f"Song   : {ini_info.get('name', song_dir.name)}")
print(f"Artist : {ini_info.get('artist', 'N/A')}")
print(f"Audio  : {audio_path}")
print(f"Chart  : {chart_path}")
print()

# ── Parse the reference chart ────────────────────────────────────────────────
# CloneHeroChartParser takes file content via from_file(); parse() does not exist.
parser = CloneHeroChartParser.from_file(str(chart_path))

# Song metadata (dict)
song_meta = parser.get_song_metadata() or {}
resolution = int(song_meta.get("Resolution", 192))
offset     = float(song_meta.get("Offset", 0.0))
print(f"Resolution : {resolution}")
print(f"Offset     : {offset}")

# SyncTrack — list of dicts with keys: position, type, data
sync_events = parser.get_sync_track() or []

# First BPM event: type == "B", data[0] is BPM * 1000
b_events = [e for e in sync_events if e.get("type") == "B" and e.get("data")]
ref_bpm = None
if b_events:
    ref_bpm = int(b_events[0]["data"][0]) / 1000.0
    print(f"BPM        : {ref_bpm:.3f}")

# Available note sections
sections = parser.get_sections()
note_sections = [s for s in sections if s not in ("Song", "SyncTrack", "Events")]
print(f"\nAvailable sections: {note_sections}")

# Count reference notes for the selected instrument/difficulty
section_key = f"{DIFFICULTY}{INSTRUMENT}"
ref_notes_raw = parser.get_parsed_section(section_key) or []
ref_notes = [e for e in ref_notes_raw if e.get("type") == "N"]
print(f"\nReference notes in [{section_key}]: {len(ref_notes)}")


Song   : La Planta
Artist : Caos
Audio  : /s/mlopezs8/CloneCharter/test_dataset/Caos - La Planta/song.ogg
Chart  : /s/mlopezs8/CloneCharter/test_dataset/Caos - La Planta/notes.chart

Resolution : 192
Offset     : 0.0
BPM        : 127.116

Available sections: ['ExpertSingle', 'ExpertDrums']

Reference notes in [ExpertSingle]: 1137


## 4 · Cargar el modelo desde checkpoint

In [4]:
from models.chart_transformer import ChartTransformer, ModelConfig
from safetensors.torch import load_file as load_safetensors

CHECKPOINT_DIR = REPO_ROOT / "model_checkpoints"

ckpt_dirs = sorted(CHECKPOINT_DIR.glob("step_*"), key=lambda p: int(p.name.split("_")[1]))
if not ckpt_dirs:
    raise FileNotFoundError(f"No checkpoints in {CHECKPOINT_DIR}")

latest = ckpt_dirs[-1]
print(f"Loading: {latest}")

cfg = ModelConfig()
pt_file = latest / "model.pt"
st_file = latest / "model.safetensors"

if pt_file.exists():
    model = ChartTransformer.load(str(pt_file), map_location=DEVICE)
elif st_file.exists():
    state = load_safetensors(str(st_file), device=DEVICE)
    model = ChartTransformer(cfg)
    # output_proj.weight is weight-tied to tok_emb.weight; absent from safetensors file.
    missing, _ = model.load_state_dict(state, strict=False)
    extra_missing = [k for k in missing if k != "output_proj.weight"]
    if extra_missing:
        raise RuntimeError(f"Unexpected missing keys: {extra_missing}")
    model.output_proj.weight = model.tok_emb.weight   # re-apply weight tying
    model.to(DEVICE)
else:
    raise FileNotFoundError(f"No weights in {latest}")

model.eval()
n_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded — {n_params / 1e6:.1f}M params on {DEVICE}")


Loading: /s/mlopezs8/CloneCharter/model_checkpoints/step_0008700


/s/mlopezs8/.local/lib/python3.12/site-packages/torch/nn/modules/transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Model loaded — 174.4M params on cuda


## 5 · Correr el pipeline completo: audio → tokens → `.chart`

`ChartGenerationPipeline` se encarga de:
1. Separación de stems con Demucs
2. Log-mel spectrogram
3. Embedding MERT
4. Estimación de BPM (o usa el del chart de referencia)
5. Encoder + beam search decoder
6. Serialización a texto `.chart`

In [5]:
from models.tokenizer import CloneHeroTokenizer
from models.mert import MERT

# ── Build pipeline without re-loading the model ──────────────────────────────
# ChartGenerationPipeline.__init__ calls torch.load() expecting a single .pt
# file, but accelerate saves a directory.  We bypass __init__ and inject the
# already-loaded model from cell 4 directly.
pipeline = object.__new__(ChartGenerationPipeline)
pipeline.device    = DEVICE
pipeline.model     = model          # loaded & weight-tied in cell 4
pipeline.tokenizer = CloneHeroTokenizer()
pipeline.mert      = MERT()         # only needed for audio → embedding; loaded lazily

override_bpm = ref_bpm  # set to None to trigger auto-detection

print(f"Generating chart for : {ini_info.get('name', song_dir.name)}")
print(f"Instrument / Difficulty : {INSTRUMENT} / {DIFFICULTY}")
print(f"BPM override : {override_bpm}")
print()

generated_chart = pipeline.generate(
    audio_path=str(audio_path),
    instrument=INSTRUMENT,
    difficulty=DIFFICULTY,
    bpm=override_bpm,
    time_signature=4,
    resolution=resolution,
    offset=offset,
    beam_size=4,
    max_new_tokens=2044,
)

out_path = song_dir / f"generated_{INSTRUMENT}_{DIFFICULTY}.chart"
out_path.write_text(generated_chart, encoding="utf-8")
print(f"\nChart saved to: {out_path}")
print(f"\n--- Preview (first 60 lines) ---")
print("\n".join(generated_chart.splitlines()[:600]))


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.6.0+cu124).


Generating chart for : La Planta
Instrument / Difficulty : Single / Expert
BPM override : 127.116

Separating audio stems...
Separando audio con modelo: htdemucs
Selected model is a bag of 1 models. You will see that many progress bars per track.
Separated tracks will be stored in /tmp/pbs.6180576.hpcq/tmpteaue_2b/htdemucs
Separating track /tmp/pbs.6180576.hpcq/tmpteaue_2b/input_audio.wav


100%|████████████████████████████████████████████████████████████████████████| 245.7/245.7 [00:42<00:00,  5.74seconds/s]


Separación completada exitosamente
Extracting log-mel spectrogram...
Extracting MERT embeddings...
  520 tick windows → token budget 2044
Encoding...
Generating chart tokens (1 pass, budget=2044)...
  2047 raw → 0 note tokens
  First 60 raw: ['<BOS>', '<Guitar>', '<Expert>', '<EOS>', '<BOS>', '<Guitar>', '<Expert>', '<EOS>', '<BOS>', '<Guitar>', '<Expert>', '<EOS>', '<BOS>', '<Guitar>', '<Expert>', '<EOS>', '<BOS>', '<Guitar>', '<Expert>', '<EOS>', '<BOS>', '<Guitar>', '<Expert>', '<EOS>', '<BOS>', '<Guitar>', '<Expert>', '<EOS>', '<BOS>', '<Guitar>', '<Expert>', '<EOS>', '<BOS>', '<Guitar>', '<Expert>', '<EOS>', '<BOS>', '<Guitar>', '<Expert>', '<EOS>', '<BOS>', '<Guitar>', '<Expert>', '<EOS>', '<BOS>', '<Guitar>', '<Expert>', '<EOS>', '<BOS>', '<Guitar>', '<Expert>', '<EOS>', '<BOS>', '<Guitar>', '<Expert>', '<EOS>', '<BOS>', '<Guitar>', '<Expert>', '<EOS>']
  Token distribution: [('<BOS>', 512), ('<Guitar>', 512), ('<Expert>', 512), ('<EOS>', 511)]
Writing chart...

Chart saved to: 

## 6 · Comparar notas generadas vs referencia

Contamos notas por pitch para ver qué tan cerca está la distribución del modelo respecto al chart original.

In [6]:
import re
from collections import Counter

PITCH_LABELS = {0: "Green", 1: "Red", 2: "Yellow", 3: "Blue", 4: "Orange"}

def parse_notes_from_chart_text(chart_text: str, section: str):
    """Return list of (tick, button, sustain) from a .chart section string."""
    in_section = False
    notes = []
    for line in chart_text.splitlines():
        line = line.strip()
        if line == f"[{section}]":
            in_section = True
            continue
        if in_section:
            if line in ("{", ""):
                continue
            if line == "}":
                break
            m = re.match(r"(\d+)\s*=\s*N\s+(\d+)\s+(\d+)", line)
            if m:
                notes.append((int(m.group(1)), int(m.group(2)), int(m.group(3))))
    return notes

section_key = f"{DIFFICULTY}{INSTRUMENT}"

# Reference notes: ref_notes is a list of dicts with keys position/type/data
# data[0] = button, data[1] = sustain
ref_count = Counter(
    int(e["data"][0]) for e in ref_notes if e.get("data")
)

# Generated notes from the saved chart text
gen_notes = parse_notes_from_chart_text(generated_chart, section_key)
gen_count  = Counter(b for _, b, _ in gen_notes)

print(f"=== Note count comparison — [{section_key}] ===\n")
print(f"{'Pitch':<12} {'Label':<10} {'Reference':>10} {'Generated':>10}")
print("-" * 45)
for p in sorted(set(ref_count) | set(gen_count)):
    label = PITCH_LABELS.get(p, str(p))
    print(f"{p:<12} {label:<10} {ref_count.get(p, 0):>10} {gen_count.get(p, 0):>10}")
print("-" * 45)
print(f"{'TOTAL':<22} {sum(ref_count.values()):>10} {len(gen_notes):>10}")


=== Note count comparison — [ExpertSingle] ===

Pitch        Label       Reference  Generated
---------------------------------------------
0            Green              91          0
1            Red               193          0
2            Yellow            296          0
3            Blue              295          0
4            Orange            177          0
5            5                   6          0
6            6                  79          0
---------------------------------------------
TOTAL                        1137          0


## 7 · Generar todas las canciones del test_dataset (batch)

Itera sobre las 3 canciones y guarda un chart generado por cada una.

In [8]:
results = []

for _song_dir in sorted(songs):
    audio = find_audio(_song_dir)
    if audio is None:
        print(f"[SKIP] {_song_dir.name} — no audio found")
        continue

    ini = read_song_ini(_song_dir)
    _chart_path = _song_dir / "notes.chart"

    # Parse BPM + resolution via the correct API
    try:
        _parser     = CloneHeroChartParser.from_file(str(_chart_path))
        _meta       = _parser.get_song_metadata() or {}
        _sync       = _parser.get_sync_track() or []
        _b_evts     = [e for e in _sync if e.get("type") == "B" and e.get("data")]
        bpm_override = int(_b_evts[0]["data"][0]) / 1000.0 if _b_evts else None
        res          = int(_meta.get("Resolution", 192))
        off          = float(_meta.get("Offset", 0.0))
    except Exception as e:
        print(f"[WARN] Could not parse {_chart_path}: {e}")
        bpm_override, res, off = None, 192, 0.0

    print(f"Generating: {ini.get('name', _song_dir.name)} — BPM={bpm_override}")

    try:
        chart_str = pipeline.generate(
            audio_path=str(audio),
            instrument=INSTRUMENT,
            difficulty=DIFFICULTY,
            bpm=bpm_override,
            time_signature=4,
            resolution=res,
            offset=off,
            beam_size=4,
            max_new_tokens=2044,
        )
        out = _song_dir / f"generated_{INSTRUMENT}_{DIFFICULTY}.chart"
        out.write_text(chart_str, encoding="utf-8")
        n_notes = len(parse_notes_from_chart_text(chart_str, f"{DIFFICULTY}{INSTRUMENT}"))
        results.append({"song": _song_dir.name, "notes": n_notes, "path": str(out)})
        print(f"  -> {n_notes} notes written to {out.name}\n")
    except Exception as e:
        print(f"  -> ERROR: {e}\n")
        results.append({"song": _song_dir.name, "notes": -1, "path": str(e)})

print("\n=== Summary ===")
for r in results:
    print(f"  {r['song']:<45} {r['notes']:>6} notes  ->  {Path(r['path']).name}")


Generating: La Planta — BPM=127.116
Separating audio stems...
Separando audio con modelo: htdemucs
Selected model is a bag of 1 models. You will see that many progress bars per track.
Separated tracks will be stored in /tmp/pbs.6180576.hpcq/tmp7zsyx9nd/htdemucs
Separating track /tmp/pbs.6180576.hpcq/tmp7zsyx9nd/input_audio.wav


100%|██████████████████████████████████████████████| 251.54999999999998/251.54999999999998 [00:43<00:00,  5.72seconds/s]


Separación completada exitosamente
Extracting log-mel spectrogram...
Extracting MERT embeddings...
Encoding...
Generating chart tokens...
Writing chart...
  -> 0 notes written to generated_Single_Expert.chart

Generating: El Precio de la Soledad — BPM=130.0
Separating audio stems...
Separando audio con modelo: htdemucs
Selected model is a bag of 1 models. You will see that many progress bars per track.
Separated tracks will be stored in /tmp/pbs.6180576.hpcq/tmpplhc0knj/htdemucs
Separating track /tmp/pbs.6180576.hpcq/tmpplhc0knj/input_audio.wav


100%|████████████████████████████████████████████████████████████████████████| 187.2/187.2 [00:32<00:00,  5.74seconds/s]


Separación completada exitosamente
Extracting log-mel spectrogram...
Extracting MERT embeddings...
Encoding...
Generating chart tokens...
Writing chart...
  -> 0 notes written to generated_Single_Expert.chart

Generating: El Rescate (ft. Junior H) — BPM=116.753
Separating audio stems...
Separando audio con modelo: htdemucs
Selected model is a bag of 1 models. You will see that many progress bars per track.
Separated tracks will be stored in /tmp/pbs.6180576.hpcq/tmpvjajkn3k/htdemucs
Separating track /tmp/pbs.6180576.hpcq/tmpvjajkn3k/input_audio.wav


100%|██████████████████████████████████████████████| 163.79999999999998/163.79999999999998 [00:29<00:00,  5.57seconds/s]


Separación completada exitosamente
Extracting log-mel spectrogram...
Extracting MERT embeddings...
Encoding...
Generating chart tokens...
Writing chart...
  -> 0 notes written to generated_Single_Expert.chart


=== Summary ===
  Caos - La Planta                                   0 notes  ->  generated_Single_Expert.chart
  El Precio de la Soledad                            0 notes  ->  generated_Single_Expert.chart
  Grupo Marca Registrada - El Rescate                0 notes  ->  generated_Single_Expert.chart
